# NOAA Marine Cadastre multi-port AIS gateway pipeline, 2015–2024

This notebook builds a reproducible manifest of the NOAA/USCG daily AIS archive, downloads selected archives with resume support, clips vessel-position messages around all configured major-port approaches, lets you draw one or more two-point gateway lines per port group, and converts line crossings into daily port-group activity metrics.

The analysis period is deliberately restricted to **2015-01-01 through 2024-12-31**: these ten calendar years have a consistent daily national CSV archive. Pre-2015 archives are monthly file geodatabases, and the 2025 bulk index is not yet available at the same URL. A full national pull is extremely large (NOAA reports 106 GB for 2017 alone), so the notebook defaults to a dry run and a one-archive pilot. Event-window downloads are strongly recommended before attempting the full decade.

Official sources: [NOAA vessel traffic data](https://www.coast.noaa.gov/digitalcoast/data/vesseltraffic.html), [2017 daily archive example](https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2017/index.html), and [AIS data dictionary](https://coast.noaa.gov/data/marinecadastre/ais/data-dictionary.pdf).

## 1. Install dependencies

Run this once per new notebook environment. `ipyleaflet` and `ipywidgets` provide the optional in-notebook gate drawing interface; QGIS is not required.

In [1]:
%pip install -q requests beautifulsoup4 pandas numpy pyarrow shapely duckdb tqdm ipyleaflet ipywidgets folium

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Each bounding box is only a processing pre-filter around one harbor approach; the gateway line drawn later is the actual geofence. Bounding boxes are `(min_longitude, min_latitude, max_longitude, max_latitude)` in WGS84. Nearby ports are merged only when they share a port complex/navigation corridor or form a deliberately defined regional storm-exposure cluster. A merged group may have multiple physical gate lines, whose crossings are aggregated after detection.

Merge documentation: [San Pedro Bay Port Complex](https://portoflosangeles.org/about), [Northwest Seaport Alliance](https://www.nwseaportalliance.com/service-providers/shipper-resources/tariff-notices), [Port of New York and New Jersey](https://www.panynj.gov/content/dam/port/customer-library-pdfs/port-capabilities.pdf), [Delaware River navigation channel](https://www.nap.usace.army.mil/Missions/Civil-Works/Delaware-River-Main-Channel-Deepening/), and [Galveston–Texas City–Houston channels](https://www.swg.usace.army.mil/Missions/Navigation/Hydrographic-Surveys/Galveston-Texas-City-Houston/). Miami–Port Everglades is an explicit analytical storm-exposure cluster with two separate physical gates, not a shared harbor.

In [1]:
from __future__ import annotations

import json
import re
import time
import zipfile
from datetime import date, datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

ARCHIVE_ROOT = REPO_ROOT / 'data' / 'external' / 'ais_archives'
POINT_ROOT = REPO_ROOT / 'data' / 'interim' / 'ais_points_major_ports_2015_2024_v3'
PROCESSED_ROOT = REPO_ROOT / 'data' / 'processed'
for folder in (ARCHIVE_ROOT, POINT_ROOT, PROCESSED_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

START_DATE = date(2015, 1, 1)
END_DATE = date(2024, 12, 31)

# 'full_period' selects all archives overlapping START_DATE–END_DATE.
# 'event_windows' selects only the windows listed below.
DOWNLOAD_SCOPE = 'event_windows'
EVENT_WINDOWS = {
    'harvey_2017': (date(2017, 7, 28), date(2017, 10, 27)),
}

# Optional replacement CSV ZIPs obtained from NOAA/AccessAIS. Example:
# {'url': 'https://...', 'archive_name': 'ais_replacement.zip',
#  'period_start': '2017-08-25', 'period_end': '2017-08-25'}
MANUAL_ARCHIVES = []

# Census/CBP port groups used when linking monthly trade outcomes. Texas City
# (5306) is added because its vessels share the Galveston Bay entrance.
PORT_GROUPS = {
    'san_pedro_bay': {'label': 'Los Angeles–Long Beach', 'port_codes': ['2704', '2709']},
    'oakland': {'label': 'Oakland', 'port_codes': ['2811']},
    'portland_or': {'label': 'Portland, OR', 'port_codes': ['2904']},
    'puget_sound': {'label': 'Seattle–Tacoma / NWSA', 'port_codes': ['3001', '3002']},
    'new_york_newark': {'label': 'New York–Newark', 'port_codes': ['1001', '1003']},
    'delaware_river': {'label': 'Philadelphia–Wilmington', 'port_codes': ['1101', '1103']},
    'baltimore': {'label': 'Baltimore', 'port_codes': ['1303']},
    'hampton_roads': {'label': 'Norfolk–Newport News', 'port_codes': ['1401']},
    'charleston': {'label': 'Charleston', 'port_codes': ['1601']},
    'savannah': {'label': 'Savannah', 'port_codes': ['1703']},
    'tampa': {'label': 'Tampa', 'port_codes': ['1801']},
    'jacksonville': {'label': 'Jacksonville', 'port_codes': ['1803']},
    'south_florida': {'label': 'Miami–Port Everglades', 'port_codes': ['5201', '5203']},
    'mobile': {'label': 'Mobile', 'port_codes': ['1901']},
    'new_orleans': {'label': 'New Orleans', 'port_codes': ['2002']},
    'galveston_bay': {'label': 'Houston–Texas City–Galveston', 'port_codes': ['5301', '5306', '5310']},
}

# Approximate viewing/pre-filter windows. They intentionally include water on
# both sides of each proposed gate and must be visually checked before production.
# map_center is (lat, lon); landward_reference and bbox use (lon, lat).
GATE_CONFIGS = {
    'los_angeles_entrance': {'port_group_id': 'san_pedro_bay', 'map_center': (33.72, -118.25), 'landward_reference': (-118.22, 33.76), 'bbox': (-118.38, 33.62, -118.08, 33.88)},
    'long_beach_entrance': {'port_group_id': 'san_pedro_bay', 'map_center': (33.73, -118.15), 'landward_reference': (-118.16, 33.78), 'bbox': (-118.28, 33.62, -117.98, 33.88)},
    'oakland_entrance': {'port_group_id': 'oakland', 'map_center': (37.80, -122.33), 'landward_reference': (-122.30, 37.80), 'bbox': (-122.48, 37.68, -122.18, 37.92)},
    'portland_or_river': {'port_group_id': 'portland_or', 'map_center': (45.61, -122.76), 'landward_reference': (-122.66, 45.62), 'bbox': (-123.05, 45.35, -122.45, 45.82)},
    'seattle_harbor': {'port_group_id': 'puget_sound', 'map_center': (47.60, -122.38), 'landward_reference': (-122.34, 47.60), 'bbox': (-122.52, 47.42, -122.22, 47.78)},
    'tacoma_harbor': {'port_group_id': 'puget_sound', 'map_center': (47.27, -122.47), 'landward_reference': (-122.43, 47.26), 'bbox': (-122.68, 47.10, -122.25, 47.43)},
    'new_york_harbor': {'port_group_id': 'new_york_newark', 'map_center': (40.60, -74.03), 'landward_reference': (-74.05, 40.70), 'bbox': (-74.28, 40.42, -73.72, 40.88)},
    # Southern boundary entrance for vessels reaching the port complex through Arthur Kill.
    # Reuse the already-processed New York study area so this added gate does not require
    # downloading or rebuilding the existing AIS Parquet files.
    'arthur_kill_south': {'port_group_id': 'new_york_newark', 'source_study_area': 'new_york_harbor', 'map_center': (40.51, -74.25), 'landward_reference': (-74.23, 40.56), 'bbox': (-74.30, 40.46, -74.17, 40.61)},
    'delaware_river_south': {'port_group_id': 'delaware_river', 'map_center': (39.58, -75.55), 'landward_reference': (-75.53, 39.78), 'bbox': (-75.75, 39.35, -74.92, 40.18)},
    'baltimore_harbor': {'port_group_id': 'baltimore', 'map_center': (39.20, -76.50), 'landward_reference': (-76.58, 39.26), 'bbox': (-76.78, 39.00, -76.28, 39.42)},
    'hampton_roads_entrance': {'port_group_id': 'hampton_roads', 'map_center': (36.98, -76.20), 'landward_reference': (-76.34, 36.98), 'bbox': (-76.62, 36.70, -75.72, 37.20)},
    'charleston_entrance': {'port_group_id': 'charleston', 'map_center': (32.76, -79.85), 'landward_reference': (-79.96, 32.82), 'bbox': (-80.12, 32.58, -79.58, 33.00)},
    'savannah_river_entrance': {'port_group_id': 'savannah', 'map_center': (32.05, -80.90), 'landward_reference': (-81.08, 32.10), 'bbox': (-81.28, 31.82, -80.62, 32.32)},
    'tampa_bay_entrance': {'port_group_id': 'tampa', 'map_center': (27.65, -82.65), 'landward_reference': (-82.55, 27.86), 'bbox': (-82.90, 27.35, -82.25, 28.15)},
    'jacksonville_entrance': {'port_group_id': 'jacksonville', 'map_center': (30.38, -81.42), 'landward_reference': (-81.58, 30.40), 'bbox': (-81.78, 30.12, -81.12, 30.62)},
    'miami_entrance': {'port_group_id': 'south_florida', 'map_center': (25.77, -80.14), 'landward_reference': (-80.19, 25.78), 'bbox': (-80.38, 25.58, -79.92, 25.96)},
    'port_everglades_entrance': {'port_group_id': 'south_florida', 'map_center': (26.09, -80.10), 'landward_reference': (-80.14, 26.10), 'bbox': (-80.32, 25.92, -79.90, 26.26)},
    'mobile_bay_channel': {'port_group_id': 'mobile', 'map_center': (30.60, -88.05), 'landward_reference': (-88.05, 30.69), 'bbox': (-88.30, 30.35, -87.75, 30.92)},
    'new_orleans_river': {'port_group_id': 'new_orleans', 'map_center': (29.90, -90.05), 'landward_reference': (-90.10, 29.98), 'bbox': (-90.40, 29.60, -89.65, 30.22)},
    'galveston_bay_entrance': {'port_group_id': 'galveston_bay', 'map_center': (29.34, -94.76), 'landward_reference': (-94.95, 29.55), 'bbox': (-95.20, 29.00, -94.35, 29.80)},
}
# Gates with source_study_area reuse a previously clipped study area and are not
# duplicated during future archive processing.
TARGET_BBOXES = {
    gate_id: config['bbox']
    for gate_id, config in GATE_CONFIGS.items()
    if 'source_study_area' not in config
}

# Keep only observations classified as cargo or tanker after applying both
# standard NMEA codes and the 2015-2017 four-digit AVIS codes.
KEEP_CARGO_AND_TANKER_ONLY = True

# Safety switches. Review the manifest before changing these.
RUN_DOWNLOADS = True
CONFIRM_FULL_PERIOD_DOWNLOAD = False
PROCESS_TO_PARQUET = True
DELETE_ARCHIVE_AFTER_SUCCESS = False
MAX_ARCHIVES = 3  # Keep at 1 for the first pipeline test; use None only when ready.

REQUEST_TIMEOUT_SECONDS = 180
CSV_CHUNK_ROWS = 250_000
USER_AGENT = 'supply-chain-resilience-research/0.1 (NOAA AIS downloader)'
NOAA_ROOT = 'https://coast.noaa.gov/htdata/CMSP/AISDataHandler/'

print(f'Repository: {REPO_ROOT}')
print(f'Download scope: {DOWNLOAD_SCOPE}, {START_DATE} through {END_DATE}')
print(f'Port groups: {len(PORT_GROUPS)}; physical gates: {len(GATE_CONFIGS)}')

Repository: /Users/wenyi/Documents/ChatGPT/supply-chain-resilience
Download scope: event_windows, 2015-01-01 through 2024-12-31
Port groups: 16; physical gates: 20


## 3. Discover the archive and build a manifest

The manifest is scraped from each official yearly index rather than hard-coding filenames. For 2015–2024 it should contain 3,653 daily national CSV ZIP files. These archives cannot be spatially clipped until after each ZIP has been downloaded. The completeness check below prevents a silently incomplete decade from being treated as complete data.

In [2]:
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': USER_AGENT})

DAILY_PATTERN = re.compile(r'AIS_(\d{4})_(\d{2})_(\d{2})\.zip$', re.I)
def overlaps_scope(start: date, end: date) -> bool:
    if end < START_DATE or start > END_DATE:
        return False
    if DOWNLOAD_SCOPE == 'full_period':
        return True
    if DOWNLOAD_SCOPE == 'event_windows':
        return any(not (end < left or start > right) for left, right in EVENT_WINDOWS.values())
    raise ValueError("DOWNLOAD_SCOPE must be 'full_period' or 'event_windows'")

def get_text(url: str, attempts: int = 4) -> str:
    for attempt in range(attempts):
        try:
            response = SESSION.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
            return response.text
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError('unreachable')

def discover_year(year: int) -> list[dict]:
    index_url = f'{NOAA_ROOT}{year}/index.html'
    soup = BeautifulSoup(get_text(index_url), 'html.parser')
    records: list[dict] = []
    seen: set[str] = set()
    for link in soup.find_all('a', href=True):
        href = link['href']
        if not href.lower().endswith('.zip'):
            continue
        url = urljoin(index_url, href)
        if url in seen:
            continue
        seen.add(url)
        name = Path(urlparse(url).path).name
        daily = DAILY_PATTERN.search(name)
        if daily:
            y, m, d = map(int, daily.groups())
            period_start = period_end = date(y, m, d)
            kind, zone = 'daily_csv', None
        else:
            continue
        if overlaps_scope(period_start, period_end):
            records.append({
                'year': year, 'archive_kind': kind, 'zone': zone,
                'period_start': period_start, 'period_end': period_end,
                'archive_name': name, 'url': url,
            })
    return records

manifest_records: list[dict] = []
discovery_errors: list[dict] = []
for year in tqdm(range(START_DATE.year, END_DATE.year + 1), desc='Discovering yearly indexes'):
    try:
        manifest_records.extend(discover_year(year))
    except requests.RequestException as exc:
        discovery_errors.append({'year': year, 'error': f'{type(exc).__name__}: {exc}'})

for manual in MANUAL_ARCHIVES:
    period_start = date.fromisoformat(manual['period_start'])
    period_end = date.fromisoformat(manual['period_end'])
    if overlaps_scope(period_start, period_end):
        manifest_records.append({
            'year': period_start.year, 'archive_kind': 'daily_csv', 'zone': None,
            'period_start': period_start, 'period_end': period_end,
            'archive_name': manual['archive_name'], 'url': manual['url'],
        })

manifest = (
    pd.DataFrame(manifest_records)
    .sort_values(['period_start', 'archive_name'])
    .reset_index(drop=True)
)
manifest_path = ARCHIVE_ROOT / 'noaa_ais_manifest_2015_2024.csv'
manifest.to_csv(manifest_path, index=False)

print(f'Manifest rows: {len(manifest):,}')
if discovery_errors:
    print('Yearly indexes not available; use AccessAIS/manual URLs where needed:')
    display(pd.DataFrame(discovery_errors))
manifest_summary = manifest.groupby(['year', 'archive_kind']).size().rename('archives').reset_index()
display(manifest_summary)

if DOWNLOAD_SCOPE == 'full_period':
    expected_dates = set(pd.date_range(START_DATE, END_DATE, freq='D').date)
else:
    expected_dates = set()
    for window_start, window_end in EVENT_WINDOWS.values():
        left, right = max(window_start, START_DATE), min(window_end, END_DATE)
        if left <= right:
            expected_dates.update(pd.date_range(left, right, freq='D').date)
manifest_dates = set(manifest.loc[manifest['archive_kind'].eq('daily_csv'), 'period_start'])
missing_dates = sorted(expected_dates - manifest_dates)
duplicate_dates = (
    manifest.loc[manifest['archive_kind'].eq('daily_csv'), 'period_start']
    .value_counts().loc[lambda counts: counts > 1]
)
print(f'Expected daily archives: {len(expected_dates):,}')
print(f'Missing dates: {len(missing_dates):,}; duplicate dates: {len(duplicate_dates):,}')
if missing_dates:
    display(pd.DataFrame({'missing_date': missing_dates[:20]}))
if len(manifest) != len(expected_dates) or missing_dates or len(duplicate_dates):
    print('WARNING: manifest is not a complete one-file-per-day panel; review before downloading.')
display(manifest.head())

Discovering yearly indexes:   0%|          | 0/10 [00:00<?, ?it/s]

Manifest rows: 92


,year,archive_kind,archives
0,2017,daily_csv,92


Expected daily archives: 92
Missing dates: 0; duplicate dates: 0


,year,archive_kind,zone,period_start,period_end,archive_name,url
0,2017,daily_csv,None,2017-07-28,2017-07-28,AIS_2017_07_28.zip,https://coast.noaa.gov/htdata/CMSP/AISDataHand...
1,2017,daily_csv,None,2017-07-29,2017-07-29,AIS_2017_07_29.zip,https://coast.noaa.gov/htdata/CMSP/AISDataHand...
2,2017,daily_csv,None,2017-07-30,2017-07-30,AIS_2017_07_30.zip,https://coast.noaa.gov/htdata/CMSP/AISDataHand...
3,2017,daily_csv,None,2017-07-31,2017-07-31,AIS_2017_07_31.zip,https://coast.noaa.gov/htdata/CMSP/AISDataHand...
4,2017,daily_csv,None,2017-08-01,2017-08-01,AIS_2017_08_01.zip,https://coast.noaa.gov/htdata/CMSP/AISDataHand...


### Optional size check

A full size check sends one HTTP HEAD request per archive and can itself take time. Run it on a pilot subset first. Reported server sizes may be missing.

In [6]:
def remote_size(url: str) -> int | None:
    try:
        response = SESSION.head(
            url,
            allow_redirects=True,
            timeout=(5, 15),  # connect timeout, read timeout
        )
        response.raise_for_status()
        value = response.headers.get("Content-Length")
        return int(value) if value and value.isdigit() else None
    except requests.RequestException as exc:
        print(f"Size check failed for {url}: {exc}")
        return None

size_sample = manifest.head(3).copy()
size_sample["bytes"] = [
    remote_size(url)
    for url in tqdm(size_sample["url"], desc="Checking archive sizes")
]
size_sample["GiB"] = size_sample["bytes"].div(1024**3)
display(size_sample[["archive_name", "archive_kind", "GiB", "url"]])

Checking archive sizes:   0%|          | 0/3 [00:00<?, ?it/s]

,archive_name,archive_kind,GiB,url
0,AIS_2017_07_28.zip,daily_csv,0.332406,https://coast.noaa.gov/htdata/CMSP/AISDataHand...
1,AIS_2017_07_29.zip,daily_csv,0.337601,https://coast.noaa.gov/htdata/CMSP/AISDataHand...
2,AIS_2017_07_30.zip,daily_csv,0.331178,https://coast.noaa.gov/htdata/CMSP/AISDataHand...


## 4. Resumable archive downloads

Completed ZIP files and partially downloaded `.part` files are reused on restart. ZIP integrity is checked before processing.

In [7]:
def download_archive(record: dict, attempts: int = 4) -> Path:
    destination_dir = ARCHIVE_ROOT / f"year={record['year']}"
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / record['archive_name']
    partial = destination.with_suffix(destination.suffix + '.part')

    if destination.exists() and destination.stat().st_size > 0:
        if zipfile.is_zipfile(destination):
            return destination
        raise RuntimeError(f'Existing file is not a valid ZIP: {destination}')

    for attempt in range(attempts):
        existing = partial.stat().st_size if partial.exists() else 0
        headers = {'Range': f'bytes={existing}-'} if existing else {}
        try:
            with SESSION.get(
                record['url'], headers=headers, stream=True,
                timeout=REQUEST_TIMEOUT_SECONDS,
            ) as response:
                response.raise_for_status()
                if existing and response.status_code != 206:
                    partial.unlink(missing_ok=True)
                    existing = 0
                mode = 'ab' if existing and response.status_code == 206 else 'wb'
                remaining = int(response.headers.get('Content-Length', 0))
                with partial.open(mode) as handle, tqdm(
                    total=existing + remaining if remaining else None,
                    initial=existing, unit='B', unit_scale=True,
                    desc=record['archive_name'], leave=False,
                ) as progress:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                            progress.update(len(chunk))
            partial.replace(destination)
            if not zipfile.is_zipfile(destination):
                raise RuntimeError(f'Downloaded file is not a valid ZIP: {destination}')
            return destination
        except (requests.RequestException, OSError, RuntimeError):
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError('unreachable')

## 5. Normalize and spatially clip AIS messages

The output retains point-level timestamps so it can later be aggregated hourly or daily. `event_date` is UTC. Vessel groups combine standard NMEA codes with NOAA's four-digit AVIS service codes used in 2015–2017; `Cargo` is used as a fallback when it contains a recognizable 70–89 code. NOAA changed vessel-type enrichment over time, so always report this classification rule and inspect the yearly unknown share.

In [8]:
OUTPUT_COLUMNS = [
    'MMSI', 'BaseDateTime', 'event_date', 'LAT', 'LON', 'SOG', 'COG',
    'Heading', 'VesselName', 'IMO', 'CallSign', 'VesselType', 'Status',
    'Length', 'Width', 'Draft', 'Cargo', 'TransceiverClass',
    'vessel_group', 'study_area',
]
NUMERIC_FLOAT_COLUMNS = ['LAT', 'LON', 'SOG', 'COG', 'Heading', 'Length', 'Width', 'Draft']
STRING_COLUMNS = ['MMSI', 'VesselName', 'IMO', 'CallSign', 'Status', 'TransceiverClass']
ALIASES = {
    'mmsi': 'MMSI', 'basedatetime': 'BaseDateTime', 'base_date_time': 'BaseDateTime',
    'latitude': 'LAT', 'lat': 'LAT', 'longitude': 'LON', 'lon': 'LON',
    'sog': 'SOG', 'cog': 'COG', 'heading': 'Heading',
    'vesselname': 'VesselName', 'vessel_name': 'VesselName',
    'imo': 'IMO', 'callsign': 'CallSign', 'call_sign': 'CallSign',
    'vesseltype': 'VesselType', 'vessel_type': 'VesselType',
    'status': 'Status', 'length': 'Length', 'width': 'Width',
    'draft': 'Draft', 'cargo': 'Cargo', 'transceiverclass': 'TransceiverClass',
}

def canonicalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for column in frame.columns:
        key = re.sub(r'[^a-z0-9_]', '', str(column).strip().lower())
        if key in ALIASES:
            rename[column] = ALIASES[key]
    return frame.rename(columns=rename)

def clip_and_normalize(frame: pd.DataFrame) -> pd.DataFrame:
    frame = canonicalize_columns(frame.copy())
    if 'LAT' not in frame or 'LON' not in frame:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    frame['LAT'] = pd.to_numeric(frame['LAT'], errors='coerce')
    frame['LON'] = pd.to_numeric(frame['LON'], errors='coerce')
    frame = frame[frame['LAT'].between(-90, 90) & frame['LON'].between(-180, 180)]

    clipped = []
    for area, (min_lon, min_lat, max_lon, max_lat) in TARGET_BBOXES.items():
        keep = (
            frame['LON'].between(min_lon, max_lon)
            & frame['LAT'].between(min_lat, max_lat)
        )
        if keep.any():
            part = frame.loc[keep].copy()
            part['study_area'] = area
            clipped.append(part)
    if not clipped:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    frame = pd.concat(clipped, ignore_index=True)

    for column in ('VesselType', 'Cargo'):
        if column not in frame:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype('Int64')

    # NOAA recommended groups: standard NMEA 70–79/80–89 plus the
    # four-digit AVIS codes that appear directly in 2015–2017.
    cargo_codes_avis = {1003, 1004, 1016}
    tanker_codes_avis = {1017, 1024}
    cargo_mask = (
        frame['VesselType'].between(70, 79)
        | frame['VesselType'].isin(cargo_codes_avis)
        | frame['Cargo'].between(70, 79)
    )
    tanker_mask = (
        frame['VesselType'].between(80, 89)
        | frame['VesselType'].isin(tanker_codes_avis)
        | frame['Cargo'].between(80, 89)
    )
    frame['vessel_group'] = pd.Series('unknown', index=frame.index, dtype='string')
    frame.loc[cargo_mask, 'vessel_group'] = 'cargo'
    frame.loc[tanker_mask, 'vessel_group'] = 'tanker'
    if KEEP_CARGO_AND_TANKER_ONLY:
        frame = frame[frame['vessel_group'].isin(['cargo', 'tanker'])]

    if 'BaseDateTime' not in frame:
        frame['BaseDateTime'] = pd.NaT
    frame['BaseDateTime'] = pd.to_datetime(frame['BaseDateTime'], errors='coerce', utc=True)
    frame['event_date'] = frame['BaseDateTime'].dt.date

    for column in NUMERIC_FLOAT_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype('Float64')
    for column in STRING_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
        frame[column] = frame[column].astype('string')
    for column in OUTPUT_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
    return frame[OUTPUT_COLUMNS].reset_index(drop=True)

def write_frames_to_parquet(frames, output_path: Path) -> int:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix('.parquet.part')
    writer = None
    rows = 0
    try:
        for frame in frames:
            normalized = clip_and_normalize(frame)
            if normalized.empty:
                continue
            table = pa.Table.from_pandas(normalized, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression='zstd')
            writer.write_table(table)
            rows += len(normalized)
    finally:
        if writer is not None:
            writer.close()
    if rows:
        temporary.replace(output_path)
    else:
        temporary.unlink(missing_ok=True)
    return rows

## 6. Reader for daily CSV ZIPs

In [9]:
def csv_frames_from_zip(archive_path: Path):
    with zipfile.ZipFile(archive_path) as bundle:
        members = [name for name in bundle.namelist() if name.lower().endswith('.csv')]
        if not members:
            raise RuntimeError(f'No CSV member found in {archive_path}')
        for member in members:
            with bundle.open(member) as handle:
                yield from pd.read_csv(
                    handle, chunksize=CSV_CHUNK_ROWS, low_memory=False,
                    on_bad_lines='skip',
                )

def process_archive(record: dict, archive_path: Path) -> tuple[Path, int]:
    output_dir = POINT_ROOT / f"year={record['year']}"
    output_name = re.sub(r'\.gdb|\.zip|\.+$', '', record['archive_name'], flags=re.I) + '.parquet'
    output_path = output_dir / output_name
    if output_path.exists():
        metadata = pq.read_metadata(output_path)
        return output_path, metadata.num_rows
    if record['archive_kind'] != 'daily_csv':
        raise ValueError(f"Expected daily_csv, got: {record['archive_kind']}")
    rows = write_frames_to_parquet(csv_frames_from_zip(archive_path), output_path)
    return output_path, rows

## 7. Run the resumable pipeline

Recommended first run: set `DOWNLOAD_SCOPE = 'event_windows'`, `RUN_DOWNLOADS = True`, and keep `MAX_ARCHIVES = 1`. After verifying one ZIP and Parquet output, increase the limit to 3. For all configured event windows use `MAX_ARCHIVES = None`. Only attempt all 3,653 days after setting `DOWNLOAD_SCOPE = 'full_period'`, `CONFIRM_FULL_PERIOD_DOWNLOAD = True`, and `MAX_ARCHIVES = None`. The pipeline resumes existing valid ZIP/Parquet outputs.

In [10]:
def run_pipeline(manifest_frame: pd.DataFrame) -> pd.DataFrame:
    if not RUN_DOWNLOADS:
        print('Dry run only. Set RUN_DOWNLOADS=True after reviewing the manifest.')
        return pd.DataFrame()
    if DOWNLOAD_SCOPE == 'full_period' and not CONFIRM_FULL_PERIOD_DOWNLOAD:
        raise RuntimeError(
            'Full-period download is selected. Set CONFIRM_FULL_PERIOD_DOWNLOAD=True '
            'only after checking storage, bandwidth, and runtime.'
        )

    # Re-filter here so a stale manifest built under an earlier scope cannot
    # accidentally download 2015 rows after the configuration has changed.
    scope_mask = [
        overlaps_scope(pd.Timestamp(start).date(), pd.Timestamp(end).date())
        for start, end in zip(manifest_frame['period_start'], manifest_frame['period_end'])
    ]
    scope_manifest = (
        manifest_frame.loc[scope_mask]
        .sort_values(['period_start', 'archive_name'])
        .reset_index(drop=True)
    )
    if scope_manifest.empty:
        raise RuntimeError('No manifest rows match the current scope. Rerun section 3.')
    selected = scope_manifest if MAX_ARCHIVES is None else scope_manifest.head(MAX_ARCHIVES)
    print(
        f'Selected {len(selected):,} of {len(scope_manifest):,} in-scope archives: '
        f"{selected['period_start'].min()} through {selected['period_end'].max()}"
    )
    statuses = []
    for record in tqdm(selected.to_dict('records'), total=len(selected), desc='AIS archives'):
        started = datetime.now().isoformat(timespec='seconds')
        status = {
            'archive_name': record['archive_name'], 'year': record['year'],
            'url': record['url'], 'started_at': started,
            'status': 'pending', 'rows': None, 'output_path': None, 'error': None,
        }
        try:
            archive_path = download_archive(record)
            status['status'] = 'downloaded'
            if PROCESS_TO_PARQUET:
                output_path, rows = process_archive(record, archive_path)
                status.update(status='processed', rows=rows, output_path=str(output_path))
                if DELETE_ARCHIVE_AFTER_SUCCESS and output_path.exists():
                    archive_path.unlink()
                    status['status'] = 'processed_archive_deleted'
        except Exception as exc:
            status.update(status='failed', error=f'{type(exc).__name__}: {exc}')
        statuses.append(status)
        pd.DataFrame(statuses).to_csv(ARCHIVE_ROOT / 'download_status_2015_2024.csv', index=False)
    return pd.DataFrame(statuses)

download_status = run_pipeline(manifest)
if not download_status.empty:
    display(download_status.tail())
    display(download_status['status'].value_counts(dropna=False))

Selected 3 of 92 in-scope archives: 2017-07-28 through 2017-07-30


AIS archives:   0%|          | 0/3 [00:00<?, ?it/s]

,archive_name,year,url,started_at,status,rows,output_path,error
0,AIS_2017_07_28.zip,2017,https://coast.noaa.gov/htdata/CMSP/AISDataHand...,2026-08-16T13:53:52,processed,277963,/Users/wenyi/Documents/ChatGPT/supply-chain-re...,None
1,AIS_2017_07_29.zip,2017,https://coast.noaa.gov/htdata/CMSP/AISDataHand...,2026-08-16T13:53:52,processed,274906,/Users/wenyi/Documents/ChatGPT/supply-chain-re...,None
2,AIS_2017_07_30.zip,2017,https://coast.noaa.gov/htdata/CMSP/AISDataHand...,2026-08-16T13:53:52,processed,256418,/Users/wenyi/Documents/ChatGPT/supply-chain-re...,None


status
processed    3
Name: count, dtype: int64

## 8. Build a daily activity diagnostic

This diagnostic counts messages and unique MMSIs inside each gate-specific pre-filter box; it is not a port-call series. Use it to detect receiver/data gaps before interpreting a storm effect. A simultaneous collapse in message counts across nearby study areas and both vessel groups may indicate coverage loss rather than a port closure.

In [4]:
parquet_files = sorted(POINT_ROOT.glob('year=*/*.parquet'))
if not parquet_files:
    print('No processed AIS Parquet files found yet.')
else:
    parquet_glob = str(POINT_ROOT / 'year=*' / '*.parquet').replace("'", "''")
    daily_activity = duckdb.sql(f"""
        SELECT
            event_date,
            study_area,
            vessel_group,
            count(*) AS message_count,
            count(DISTINCT MMSI) AS unique_vessels,
            avg(SOG) AS mean_speed_knots
        FROM read_parquet('{parquet_glob}', union_by_name=true)
        WHERE event_date IS NOT NULL
        GROUP BY 1, 2, 3
        ORDER BY 1, 2, 3
    """).df()
    daily_output = PROCESSED_ROOT / 'ais_daily_activity_2015_2024.parquet'
    daily_activity.to_parquet(daily_output, index=False)
    print(f'Wrote {len(daily_activity):,} rows to {daily_output}')
    display(daily_activity.tail())

Wrote 209 rows to /Users/wenyi/Documents/ChatGPT/supply-chain-resilience/data/processed/ais_daily_activity_2015_2024.parquet


,event_date,study_area,vessel_group,message_count,unique_vessels,mean_speed_knots
204,2017-07-30,seattle_harbor,cargo,1807,8,3.074765
205,2017-07-30,seattle_harbor,tanker,244,1,0.266393
206,2017-07-30,tacoma_harbor,cargo,2040,6,0.987598
207,2017-07-30,tampa_bay_entrance,cargo,6825,12,1.023751
208,2017-07-30,tampa_bay_entrance,tanker,1583,4,2.643399


## 9. Draw each physical gateway line (QGIS optional)

Run this after processing at least one representative day. Set `ACTIVE_GATE_ID`, run the cell, and use the polyline tool to draw a short, straight, **two-point line across that gate's navigable channel**, approximately perpendicular to vessel traffic. Double-click to finish. Repeat for every entry in `GATE_CONFIGS`. Each line is saved under `data/geofences/major_port_gates/`. Redrawing a gate replaces only that gate's file.

Each configuration has a `landward_reference` point on the port side of its gate. It lets the code label seaward-to-landward crossings as inbound regardless of endpoint order. Groups such as San Pedro Bay, Puget Sound, and South Florida have two physical gates; their events are aggregated to one `port_group_id` after crossing detection. Approximate centers and boxes are starting aids, not validated navigational boundaries.

In [12]:
from ipyleaflet import DrawControl, GeoJSON, LayersControl, Map, Marker, basemaps

GEOFENCE_ROOT = REPO_ROOT / 'data' / 'geofences' / 'major_port_gates'
GEOFENCE_ROOT.mkdir(parents=True, exist_ok=True)
drawn_gate_ids = {
    gate_id
    for gate_id in GATE_CONFIGS
    if (GEOFENCE_ROOT / f'{gate_id}.geojson').exists()
}
missing_gate_ids = [
    gate_id for gate_id in GATE_CONFIGS
    if gate_id not in drawn_gate_ids
]

if not missing_gate_ids:
    raise RuntimeError('All gateway lines have already been drawn.')

ACTIVE_GATE_ID = missing_gate_ids[0]

if ACTIVE_GATE_ID not in GATE_CONFIGS:
    raise KeyError(f'Unknown gate: {ACTIVE_GATE_ID}')
ACTIVE_GATE = GATE_CONFIGS[ACTIVE_GATE_ID]
PORT_GROUP_ID = ACTIVE_GATE['port_group_id']
GATE_PATH = GEOFENCE_ROOT / f'{ACTIVE_GATE_ID}.geojson'
MAP_CENTER = ACTIVE_GATE['map_center']
LANDWARD_REFERENCE = ACTIVE_GATE['landward_reference']
SOURCE_STUDY_AREA = ACTIVE_GATE.get('source_study_area', ACTIVE_GATE_ID)

gateway_map = Map(center=MAP_CENTER, zoom=10, basemap=basemaps.CartoDB.Positron)
gateway_map.add(Marker(location=(LANDWARD_REFERENCE[1], LANDWARD_REFERENCE[0]), title='Landward reference'))

sample_candidates = sorted((POINT_ROOT / 'year=2017').glob('AIS_2017_08_*.parquet'))[:3]
if not sample_candidates:
    sample_candidates = parquet_files[:3]
if sample_candidates:
    sample_parts = [pd.read_parquet(path, columns=['LON', 'LAT', 'vessel_group', 'study_area']) for path in sample_candidates]
    sample_points = pd.concat(sample_parts, ignore_index=True)
    sample_points = sample_points[sample_points['study_area'].eq(SOURCE_STUDY_AREA)].dropna(subset=['LON', 'LAT'])
    min_lon, min_lat, max_lon, max_lat = ACTIVE_GATE['bbox']
    sample_points = sample_points[
        sample_points['LON'].between(min_lon, max_lon)
        & sample_points['LAT'].between(min_lat, max_lat)
    ]
    if len(sample_points) > 5_000:
        sample_points = sample_points.sample(5_000, random_state=42)
    point_geojson = {
        'type': 'FeatureCollection',
        'features': [
            {
                'type': 'Feature',
                'geometry': {'type': 'Point', 'coordinates': [row.LON, row.LAT]},
                'properties': {'vessel_group': row.vessel_group},
            }
            for row in sample_points.itertuples(index=False)
        ],
    }
    gateway_map.add(GeoJSON(data=point_geojson, point_style={'radius': 2, 'color': '#2b8cbe', 'fillOpacity': 0.5}, name='AIS sample'))
else:
    print('No Parquet sample yet. You may draw on the basemap now or process pilot AIS days first.')

draw_control = DrawControl(
    polyline={'shapeOptions': {'color': '#d73027', 'weight': 5}},
    polygon={}, rectangle={}, circle={}, circlemarker={}, marker={},
)

def save_gateway_line(_target, action, geo_json):
    if action not in {'created', 'edited'}:
        return
    geometry = geo_json.get('geometry', {})
    if geometry.get('type') != 'LineString':
        print('Please draw a polyline, not a point or polygon.')
        return
    coordinates = geometry.get('coordinates', [])
    if len(coordinates) < 2:
        print('The gate needs two endpoints.')
        return
    straight_gate = [coordinates[0], coordinates[-1]]
    feature_collection = {
        'type': 'FeatureCollection',
        'features': [{
            'type': 'Feature',
            'properties': {'gate_id': ACTIVE_GATE_ID, 'port_group_id': PORT_GROUP_ID},
            'geometry': {'type': 'LineString', 'coordinates': straight_gate},
        }],
    }
    GATE_PATH.write_text(json.dumps(feature_collection, indent=2), encoding='utf-8')
    print(f'Saved {ACTIVE_GATE_ID} ({PORT_GROUP_ID}) to {GATE_PATH}')
    print(f'Gate coordinates: {straight_gate}')

draw_control.on_draw(save_gateway_line)
gateway_map.add(draw_control)
gateway_map.add(LayersControl())
drawn_gate_ids = [gate_id for gate_id in GATE_CONFIGS if (GEOFENCE_ROOT / f'{gate_id}.geojson').exists()]
missing_gate_ids = [gate_id for gate_id in GATE_CONFIGS if gate_id not in drawn_gate_ids]
print(f'Active gate: {ACTIVE_GATE_ID}; drawn {len(drawn_gate_ids)}/{len(GATE_CONFIGS)}')
print(f'Missing gates: {missing_gate_ids}')
gateway_map

All gateway lines have already been drawn.


NameError: name 'stop' is not defined

### 9B. Static-map fallback if `jupyter-leaflet` fails

This Folium map does not require the Jupyter widget extension. Click two endpoints on opposite sides of the channel to display their `(latitude, longitude)` values. Copy them into `MANUAL_GATE_LATLON` and rerun this cell; it saves the same gate GeoJSON consumed by Section 10.

In [31]:
import folium
from folium.plugins import FastMarkerCluster

# Replace None with exactly two popup coordinates, in (latitude, longitude) order.
# Example: MANUAL_GATE_LATLON = [(29.30, -94.79), (29.36, -94.72)]
MANUAL_GATE_LATLON = None

fallback_map = folium.Map(location=list(MAP_CENTER), zoom_start=10, tiles='CartoDB positron')
folium.LatLngPopup().add_to(fallback_map)
folium.Marker(
    [LANDWARD_REFERENCE[1], LANDWARD_REFERENCE[0]],
    tooltip='Landward reference', icon=folium.Icon(color='green'),
).add_to(fallback_map)
if 'sample_points' in globals() and not sample_points.empty:
    FastMarkerCluster(sample_points[['LAT', 'LON']].dropna().values.tolist()).add_to(fallback_map)

if MANUAL_GATE_LATLON is not None:
    if len(MANUAL_GATE_LATLON) != 2:
        raise ValueError('MANUAL_GATE_LATLON must contain exactly two (latitude, longitude) points.')
    straight_gate = [[lon, lat] for lat, lon in MANUAL_GATE_LATLON]
    feature_collection = {
        'type': 'FeatureCollection',
        'features': [{
            'type': 'Feature',
            'properties': {'gate_id': ACTIVE_GATE_ID, 'port_group_id': PORT_GROUP_ID},
            'geometry': {'type': 'LineString', 'coordinates': straight_gate},
        }],
    }
    GATE_PATH.write_text(json.dumps(feature_collection, indent=2), encoding='utf-8')
    folium.PolyLine(MANUAL_GATE_LATLON, color='red', weight=5, tooltip=ACTIVE_GATE_ID).add_to(fallback_map)
    print(f'Saved manual gate to {GATE_PATH}')
else:
    print('Click two points, copy their (lat, lon) values into MANUAL_GATE_LATLON, and rerun.')

if GATE_PATH.exists() and MANUAL_GATE_LATLON is None:
    saved = json.loads(GATE_PATH.read_text(encoding='utf-8'))['features'][0]['geometry']['coordinates']
    saved_latlon = [(lat, lon) for lon, lat in (saved[0], saved[-1])]
    folium.PolyLine(saved_latlon, color='red', weight=5, tooltip=f'Existing: {ACTIVE_GATE_ID}').add_to(fallback_map)
fallback_map

Click two points, copy their (lat, lon) values into MANUAL_GATE_LATLON, and rerun.


## 10. Detect gateway crossings and build daily metrics

For each physical gate and MMSI, adjacent observations form a short track segment. A qualifying event must cross the finite gate line, move from one side to the other, have no more than `MAX_TRACK_GAP_MINUTES` between observations, and be separated from the vessel's previous accepted event at that gate by `DEBOUNCE_HOURS`. The algorithm reads each daily Parquet once, processes all drawn gates, and carries gate-specific MMSI state across midnight.

Physical-gate events are then aggregated to `port_group_id`. The resulting metrics are `inbound_cargo`, `inbound_tanker`, `outbound` (cargo plus tanker), and `unique_vessels`. `geofence_complete` shows whether every configured gate for that group has been drawn. Dates are generated only for successfully processed daily Parquet files; an undownloaded day is never silently converted to zero.

In [7]:
MAX_TRACK_GAP_MINUTES = 60
DEBOUNCE_HOURS = 6
MIN_CROSSING_SOG_KNOTS = 0.5
TRACK_COLUMNS = ['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'vessel_group', 'study_area']

def load_gate_coordinates(path: Path) -> tuple[tuple[float, float], tuple[float, float]]:
    payload = json.loads(path.read_text(encoding='utf-8'))
    feature = payload['features'][0] if payload.get('type') == 'FeatureCollection' else payload
    geometry = feature['geometry'] if feature.get('type') == 'Feature' else feature
    if geometry.get('type') != 'LineString' or len(geometry.get('coordinates', [])) < 2:
        raise ValueError('Gateway GeoJSON must contain a LineString with at least two coordinates.')
    coordinates = geometry['coordinates']
    return tuple(coordinates[0]), tuple(coordinates[-1])

def orientation(ax, ay, bx, by, px, py):
    return (bx - ax) * (py - ay) - (by - ay) * (px - ax)

def archive_date_from_name(path: Path) -> date | None:
    match = re.search(r'AIS_(\d{4})_(\d{2})_(\d{2})', path.name, re.I)
    return date(*map(int, match.groups())) if match else None

def build_major_port_crossings(paths: list[Path], gate_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    gate_metadata = {}
    for gate_id, config in GATE_CONFIGS.items():
        gate_path = gate_root / f'{gate_id}.geojson'
        if not gate_path.exists():
            continue
        (ax, ay), (bx, by) = load_gate_coordinates(gate_path)
        reference_side = orientation(ax, ay, bx, by, *config['landward_reference'])
        if abs(reference_side) < 1e-12:
            raise ValueError(f'landward_reference lies on gate {gate_id}.')
        gate_metadata[gate_id] = {**config, 'line': (ax, ay, bx, by), 'reference_side': reference_side}
    if not gate_metadata:
        raise RuntimeError('No gate GeoJSON files found. Draw at least one gate in section 9.')

    states = {
        gate_id: {'last_points': pd.DataFrame(columns=TRACK_COLUMNS), 'last_events': {}}
        for gate_id in gate_metadata
    }
    accepted_events: list[dict] = []
    covered_dates: set[date] = set()

    for path in tqdm(sorted(paths), desc='Detecting all gate crossings'):
        archive_date = archive_date_from_name(path)
        if archive_date is not None:
            covered_dates.add(archive_date)
        all_points = pd.read_parquet(path, columns=TRACK_COLUMNS)
        all_points['BaseDateTime'] = pd.to_datetime(all_points['BaseDateTime'], errors='coerce', utc=True)
        for column in ['LAT', 'LON', 'SOG']:
            all_points[column] = pd.to_numeric(all_points[column], errors='coerce')
        all_points['MMSI'] = all_points['MMSI'].astype('string')
        all_points = all_points.dropna(subset=['MMSI', 'BaseDateTime', 'LAT', 'LON'])
        all_points = all_points[all_points['vessel_group'].isin(['cargo', 'tanker'])]

        for gate_id, metadata in gate_metadata.items():
            source_study_area = metadata.get('source_study_area', gate_id)
            current = all_points[all_points['study_area'].eq(source_study_area)].copy()
            min_lon, min_lat, max_lon, max_lat = metadata['bbox']
            current = current[
                current['LON'].between(min_lon, max_lon)
                & current['LAT'].between(min_lat, max_lat)
            ]
            if current.empty:
                continue
            state = states[gate_id]
            ax, ay, bx, by = metadata['line']
            reference_side = metadata['reference_side']
            current = current.sort_values(['MMSI', 'BaseDateTime']).drop_duplicates(['MMSI', 'BaseDateTime'], keep='last')
            current['_current_file'] = True
            if state['last_points'].empty:
                combined = current.copy()
            else:
                prior = state['last_points'].copy()
                prior['_current_file'] = False
                combined = pd.concat([prior, current], ignore_index=True)
            combined = combined.sort_values(['MMSI', 'BaseDateTime'])
            grouped = combined.groupby('MMSI', sort=False)
            for column in ['BaseDateTime', 'LAT', 'LON']:
                combined[f'prev_{column}'] = grouped[column].shift(1)

            candidates = combined[combined['_current_file'] & combined['prev_BaseDateTime'].notna()].copy()
            gap_minutes = (candidates['BaseDateTime'] - candidates['prev_BaseDateTime']).dt.total_seconds() / 60
            candidates = candidates[gap_minutes.gt(0) & gap_minutes.le(MAX_TRACK_GAP_MINUTES)]
            candidates = candidates[candidates['SOG'].isna() | candidates['SOG'].ge(MIN_CROSSING_SOG_KNOTS)]

            if not candidates.empty:
                prev_side = orientation(ax, ay, bx, by, candidates['prev_LON'], candidates['prev_LAT'])
                current_side = orientation(ax, ay, bx, by, candidates['LON'], candidates['LAT'])
                track_dx = candidates['LON'] - candidates['prev_LON']
                track_dy = candidates['LAT'] - candidates['prev_LAT']
                gate_a_side = track_dx * (ay - candidates['prev_LAT']) - track_dy * (ax - candidates['prev_LON'])
                gate_b_side = track_dx * (by - candidates['prev_LAT']) - track_dy * (bx - candidates['prev_LON'])
                finite_crossing = prev_side.mul(current_side).lt(0) & gate_a_side.mul(gate_b_side).le(0)
                candidates = candidates[finite_crossing].copy()
                current_side = current_side.loc[candidates.index]
                candidates['direction'] = np.where(current_side * reference_side > 0, 'inbound', 'outbound')

                for event in candidates.sort_values('BaseDateTime').itertuples(index=False):
                    previous_event = state['last_events'].get(str(event.MMSI))
                    if previous_event is not None and event.BaseDateTime - previous_event < pd.Timedelta(hours=DEBOUNCE_HOURS):
                        continue
                    accepted_events.append({
                        'gate_id': gate_id,
                        'port_group_id': metadata['port_group_id'],
                        'MMSI': str(event.MMSI),
                        'crossing_time_utc': event.BaseDateTime,
                        'date': event.BaseDateTime.date(),
                        'direction': event.direction,
                        'vessel_group': event.vessel_group,
                        'SOG': event.SOG, 'LON': event.LON, 'LAT': event.LAT,
                    })
                    state['last_events'][str(event.MMSI)] = event.BaseDateTime

            latest = current.sort_values('BaseDateTime').groupby('MMSI', as_index=False).tail(1)[TRACK_COLUMNS]
            state['last_points'] = pd.concat([state['last_points'], latest], ignore_index=True)
            state['last_points'] = state['last_points'].sort_values('BaseDateTime').groupby('MMSI', as_index=False).tail(1)

    event_columns = ['gate_id', 'port_group_id', 'MMSI', 'crossing_time_utc', 'date', 'direction', 'vessel_group', 'SOG', 'LON', 'LAT']
    crossings = pd.DataFrame(accepted_events, columns=event_columns)
    active_groups = sorted({metadata['port_group_id'] for metadata in gate_metadata.values()})
    calendar_frame = pd.MultiIndex.from_product([active_groups, sorted(covered_dates)], names=['port_group_id', 'date']).to_frame(index=False)

    if crossings.empty:
        daily = calendar_frame.assign(inbound_cargo=0, inbound_tanker=0, outbound=0, unique_vessels=0)
    else:
        flagged = crossings.assign(
            inbound_cargo=((crossings['direction'] == 'inbound') & (crossings['vessel_group'] == 'cargo')).astype(int),
            inbound_tanker=((crossings['direction'] == 'inbound') & (crossings['vessel_group'] == 'tanker')).astype(int),
            outbound=(crossings['direction'] == 'outbound').astype(int),
        )
        counts = flagged.groupby(['port_group_id', 'date'], as_index=False).agg(
            inbound_cargo=('inbound_cargo', 'sum'), inbound_tanker=('inbound_tanker', 'sum'),
            outbound=('outbound', 'sum'), unique_vessels=('MMSI', 'nunique'),
        )
        daily = calendar_frame.merge(counts, on=['port_group_id', 'date'], how='left').fillna(0)
        for column in ['inbound_cargo', 'inbound_tanker', 'outbound', 'unique_vessels']:
            daily[column] = daily[column].astype(int)

    configured_gate_counts = pd.Series([c['port_group_id'] for c in GATE_CONFIGS.values()]).value_counts()
    drawn_gate_counts = pd.Series([m['port_group_id'] for m in gate_metadata.values()]).value_counts()
    daily['configured_gate_count'] = daily['port_group_id'].map(configured_gate_counts).astype(int)
    daily['drawn_gate_count'] = daily['port_group_id'].map(drawn_gate_counts).astype(int)
    daily['geofence_complete'] = daily['drawn_gate_count'].eq(daily['configured_gate_count'])
    daily['ais_file_available'] = 1
    daily['port_codes'] = daily['port_group_id'].map(lambda group: ','.join(PORT_GROUPS[group]['port_codes']))
    return crossings, daily

parquet_files = sorted(POINT_ROOT.glob('year=*/*.parquet'))
drawn_gates = [gate_id for gate_id in GATE_CONFIGS if (GEOFENCE_ROOT / f'{gate_id}.geojson').exists()]
if not drawn_gates:
    print(f'No gate files found under {GEOFENCE_ROOT}. Draw at least one gate in section 9.')
elif not parquet_files:
    print('No processed AIS Parquet files found. Run a pilot download first.')
else:
    major_port_crossings, port_group_daily = build_major_port_crossings(parquet_files, GEOFENCE_ROOT)
    crossing_output = PROCESSED_ROOT / 'ais_major_port_gate_crossings_2015_2024.parquet'
    daily_output = PROCESSED_ROOT / 'ais_major_port_group_daily_metrics_2015_2024.parquet'
    major_port_crossings.to_parquet(crossing_output, index=False)
    port_group_daily.to_parquet(daily_output, index=False)
    print(f'Wrote {len(major_port_crossings):,} crossings from {len(drawn_gates)} gates to {crossing_output}')
    print(f'Wrote {len(port_group_daily):,} port-group dates to {daily_output}')
    display(port_group_daily.tail())

Detecting all gate crossings:   0%|          | 0/6 [00:00<?, ?it/s]

/var/folders/pk/s51rghzs0v9grvswhrf6kp5w0000gp/T/ipykernel_59514/3837230819.py:115: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  state['last_points'] = pd.concat([state['last_points'], latest], ignore_index=True)
/var/folders/pk/s51rghzs0v9grvswhrf6kp5w0000gp/T/ipykernel_59514/3837230819.py:115: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  state['last_points'] = pd.concat([state['last_points'], latest], ignore_index=True)
/var/folders/pk/s51rghzs0v9grvswhrf6kp5w0000gp/T/ipykernel_59514/3837230

Wrote 1,346 crossings from 20 gates to /Users/wenyi/Documents/ChatGPT/supply-chain-resilience/data/processed/ais_major_port_gate_crossings_2015_2024.parquet
Wrote 96 port-group dates to /Users/wenyi/Documents/ChatGPT/supply-chain-resilience/data/processed/ais_major_port_group_daily_metrics_2015_2024.parquet


,port_group_id,date,inbound_cargo,inbound_tanker,outbound,unique_vessels,configured_gate_count,drawn_gate_count,geofence_complete,ais_file_available,port_codes
91,tampa,2015-01-02,3,0,2,5,1,1,True,1,1801
92,tampa,2015-01-03,4,0,2,5,1,1,True,1,1801
93,tampa,2017-07-28,1,1,6,8,1,1,True,1,1801
94,tampa,2017-07-29,4,0,3,7,1,1,True,1,1801
95,tampa,2017-07-30,2,3,3,8,1,1,True,1,1801


## 11. Interpretation and validation checklist

1. Plot several days of accepted crossings on the map and manually verify direction labels.
2. Compare event-period results with documented channel closure/reopening dates for the affected port group.
3. Check `ais_daily_activity_2015_2024.parquet` for receiver outages and year-to-year classification changes.
4. Treat each result as a **port-group gateway activity** measure, not import tonnage. A vessel crossing may be exporting, arriving in ballast, or calling at a different terminal within the group.
5. For monthly Census comparisons, aggregate vessel-mode outcomes with the `PORT_GROUPS` crosswalk below. Prefer `VES_VAL_MO`/`VES_WGT_MO`; use `CNT_VAL_MO`/`CNT_WGT_MO` for the container subset. Never match a merged AIS group to only one of its component Census port codes.

Keep point extracts immutable. If a bounding box, gate, landward reference, or debounce setting changes, regenerate the v3 point/crossing/daily outputs and record the parameter values used.

In [10]:
port_group_crosswalk = pd.DataFrame([
    {'port_group_id': group_id, 'port_group_label': config['label'], 'port_code': port_code}
    for group_id, config in PORT_GROUPS.items()
    for port_code in config['port_codes']
])
crosswalk_output = PROCESSED_ROOT / 'major_port_group_crosswalk.csv'
port_group_crosswalk.to_csv(crosswalk_output, index=False)
print(f'Wrote Census port-group crosswalk to {crosswalk_output}')
display(port_group_crosswalk)

Wrote Census port-group crosswalk to /Users/wenyi/Documents/ChatGPT/supply-chain-resilience/data/processed/major_port_group_crosswalk.csv


,port_group_id,port_group_label,port_code
0,san_pedro_bay,Los Angeles–Long Beach,2704
1,san_pedro_bay,Los Angeles–Long Beach,2709
2,oakland,Oakland,2811
3,portland_or,"Portland, OR",2904
4,puget_sound,Seattle–Tacoma / NWSA,3001
5,puget_sound,Seattle–Tacoma / NWSA,3002
6,new_york_newark,New York–Newark,1001
7,new_york_newark,New York–Newark,1003
8,delaware_river,Philadelphia–Wilmington,1101
9,delaware_river,Philadelphia–Wilmington,1103
